In [1]:
from enum import Enum, auto, StrEnum
from pathlib import Path
# import traceback
from PIL import Image
import math

import torch
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import datasets, transforms

from tqdm import tqdm

In [ ]:
class SinusoidalEmbedding(nn.Module):

    def __init__(self, dim: int, max_period = 10000):
        assert dim % 2 == 0, "dim must be divisible by 2"
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32) / half)
        self.register_buffer('freqs', freqs, persistent=False)


    def forward(self, t: Tensor):
        t = t.reshape(-1).float()
        args = t[:, None] * self.freqs[None, :]
        return torch.concat(torch.sin(args), torch.cos(args), dim=1)
    

def group_norm(channels: int, max_groups: int):
    groups = max_groups
    while (channels % groups) == 0:
        groups //= 2
    return nn.GroupNorm(groups, channels)



class TimeEmbedding(nn.Module):

    def __init__(self,
                 base_dim: int,
                 emb_dim: int,
                 dropout: float = 0.1):
        
        super().__init__()
        self.sinusoid = SinusoidalEmbedding(dim=base_dim)
        self.mlp = nn.Sequential(
            nn.Linear(base_dim, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim)
        )


    def forward(self, t: Tensor):

        return self.mlp(self.sinusoid(t))



class ResBlock(nn.Module):

    def __init__(self,
                 in_channel: int,
                 out_channel: int,
                 emb_dim: int,
                 dropout: float):
        super().__init__()

        self.norm1 = nn.GroupNorm(group_norm(in_channel), in_channel)

        self.conv1 = nn.Conv2d(
            in_channels=in_channel,
            out_channels=out_channel)

        self.time_proj = nn.Linear(emb_dim, out_channel)

        self.norm2 = nn.GroupNorm(group_norm(out_channel), out_channel)

        self.dropout = nn.Dropout(dropout)
        
        self.conv2 = nn.Conv2d(
            in_channels=in_channel,
            out_channels=out_channel)

        nn.init.zeros_(self.conv2.weight)
        nn.init.zeros_(self.conv2.bias)

        self.skip = (
            nn.Conv2d(
                in_channel, 
                out_channel, 
                kernel_size=1)
            
            if in_channel != out_channel 
            else 
            nn.Identity())




    def forward(self, x: Tensor, emb: Tensor):

        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(F.silu(emb))[:, :, None, None]
        h = self.conv2(self.dropout(F.silu(self.norm2(h))))
        return h + self.skip(x)




In [3]:


class UNet(nn.Module):

    def __init__(self):

        super().__init__()
 
        self.norm1 = nn.GroupNorm(8, 24)
        self.norm2 = nn.GroupNorm(32, 64)
        self.norm3 = nn.GroupNorm(32, 128)
        self.norm4 = nn.GroupNorm(32, 192)
        self.norm5 = nn.GroupNorm(32, 256)
        self.norm6 = nn.GroupNorm(32, 384)
        self.norm7 = nn.GroupNorm(32, 512)

        
        self.time_emb1 = nn.Linear(1, 24)
        self.time_emb2 = nn.Linear(1, 64)
        self.time_emb3 = nn.Linear(1, 128)
        self.time_emb4 = nn.Linear(1, 192)
        self.time_emb5 = nn.Linear(1, 256)
        self.time_emb6 = nn.Linear(1, 384)
        self.time_emb7 = nn.Linear(1, 512)

        self.proj1 = nn.Conv2d(
            in_channels=768,
            out_channels=384,
            kernel_size=3,
            padding=1,
            stride=1
        )

        self.proj2 = nn.Conv2d(
            in_channels=512,
            out_channels=256,
            kernel_size=3,
            padding=1,
            stride=1
        )


        self.proj3 = nn.Conv2d(
            in_channels=384,
            out_channels=192,
            kernel_size=3,
            padding=1,
            stride=1
        )

        self.proj4 = nn.Conv2d(
            in_channels=256,
            out_channels=128,
            kernel_size=3,
            padding=1,
            stride=1
        )



        # batch x 3 x 1024 x 1024
        # batch x 3 x 512 x 512

        self.encoder1 = nn.Conv2d(
                    in_channels=3,
                    out_channels=24,
                    kernel_size=7,
                    padding=3,
                    stride=1
                )
        
        # batch x 24 x 1024 x 1024
        # batch x 24 x 512 x 512

        self.encoder2 = nn.Conv2d(
                    in_channels=24,
                    out_channels=64,
                    kernel_size=5,
                    padding=2,
                    stride=1
                )
        
        # batch x 64 x 1024 x 1024
        # batch x 64 x 512 x 512

        self.encoder3 = nn.Conv2d(
                    in_channels=64,
                    out_channels=128,
                    kernel_size=3,
                    padding=1,
                    stride=1
                )
        
        # batch x 128 x 1024 x 1024
        # batch x 128 x 512 x 512

        self.downsamp1 = nn.Conv2d(
                    in_channels=128,
                    out_channels=192,
                    kernel_size=3,
                    padding=1,
                    stride=2
                )
        
        # batch x 192 x 512 x 512
        # batch x 192 x 256 x 256

        self.downsamp2 = nn.Conv2d(
                    in_channels=192,
                    out_channels=256,
                    kernel_size=3,
                    padding=1,
                    stride=2
                )
        
        # batch x 256 x 256 x 256
        # batch x 256 x 128 x 128

        self.downsamp3 = nn.Conv2d(
                    in_channels=256,
                    out_channels=384,
                    kernel_size=3,
                    padding=1,
                    stride=2
                )

        # batch x 384 x 128 x 128
        # batch x 384 x 64 x 64

        self.downsamp4 = nn.Conv2d(
            in_channels=384,
            out_channels=512,
            kernel_size=3,
            padding=1,
            stride=2
        )

        # batch x 512 x 64 x 64
        # batch x 512 x 32 x 32


        self.bottleneck = nn.Sequential(
            nn.GroupNorm(32, 512),
            nn.SiLU(),
            nn.Conv2d(
                in_channels=512,
                out_channels=512,
                kernel_size=3,
                padding=1,
                stride=1
            ),
            nn.GroupNorm(32, 512),
            nn.SiLU(),
            nn.Dropout2d(p=0.1),
            nn.Conv2d(
                in_channels=512,
                out_channels=512,
                kernel_size=3,
                padding=1,
                stride=1
            ),
            nn.GroupNorm(32, 512),
            nn.SiLU(),
            nn.Dropout2d(p=0.2),
            nn.Conv2d(
                in_channels=512,
                out_channels=512,
                kernel_size=3,
                padding=1,
                stride=1
            ),
            nn.GroupNorm(32, 512),
            nn.SiLU(),
            nn.Dropout2d(p=0.3),
            nn.Conv2d(
                in_channels=512,
                out_channels=512,
                kernel_size=3,
                 padding=1,
                stride=1
            ),
            nn.GroupNorm(32, 512),
            nn.SiLU(),
            nn.Dropout2d(p=0.4),
            nn.Conv2d(
                in_channels=512,
                out_channels=512,
                kernel_size=3,
                padding=1,
                stride=1
            )
        )


        self.upsamp1 = nn.ConvTranspose2d(
            in_channels=512,
            out_channels=384,
            kernel_size=3,
            padding=1,
            output_padding=1,
            stride=2
        )

        self.upsamp2 = nn.ConvTranspose2d(
            in_channels=384,
            out_channels=256,
            kernel_size=3,
            padding=1,
            output_padding=1,
            stride=2
        )

        self.upsamp3 = nn.ConvTranspose2d(
            in_channels=256,
            out_channels=192,
            kernel_size=3,
            padding=1,
            output_padding=1,
            stride=2
        )


        self.upsamp4 = nn.ConvTranspose2d(
            in_channels=192,
            out_channels=128,
            kernel_size=3,
            padding=1,
            output_padding=1,
            stride=2
        )

        self.decoder1 = nn.Conv2d(
            in_channels=128,
            out_channels=64,
            kernel_size=3,
            padding=1,
            stride=1
        )

        self.decoder2 = nn.Conv2d(
            in_channels=64,
            out_channels=24,
            kernel_size=5,
            padding=2,
            stride=1
        )

        self.decoder3 = nn.Conv2d(
             in_channels=24,
             out_channels=3,
             kernel_size=7,
             padding=3,
             stride=1
         ) 


    def forward(self, x: Tensor, t: Tensor):


        # [batch x 3 x 1024 x 1024]
        h01 = self.encoder1(x)   
        # [batch x 24 x 1024 x 1024]
        h01 = self.norm1(h01)
        h01 = F.silu(h01)  

        t1 = self.time_emb1(t) 
        t1 = t1[:, :, None, None] 

        # time-step conditioning
        h01 = h01 + t1

        # [batch x 24 x 1024 x 1024]
        h02 = self.encoder2(h01)
        # [batch x 64 x 1024 x 1024]
        h02 = self.norm2(h02)
        h02 = F.silu(h02)

        t2 = self.time_emb2(t)
        t2 = t2[:, :, None, None]

        h02 = h02 + t2

        # [batch x 64 x 1024 x 1024]
        h03 = self.encoder3(h02)
        # [batch x 128 x 1024 x 1024]
        h03 = self.norm3(h03)
        h03 = F.silu(h03)

        t3 = self.time_emb3(t)
        t3 = t3[:, :, None, None]

        h03 = h03 + t3

        # [batch x 128 x 1024 x 1024]
        h04 = self.downsamp1(h03)
        # [batch x 192 x 512 x 512]
        h04 = self.norm4(h04)
        h04 = F.silu(h04)

        t4 = self.time_emb4(t)
        t4 = t4[:, :, None, None]

        h04 = h04 + t4

        # [batch x 192 x 512 x 512]
        h05 = self.downsamp2(h04)
        # [batch x 256 x 256 x 256]
        h05 = self.norm5(h05)
        h05 = F.silu(h05)

        t5 = self.time_emb5(t)
        t5 = t5[:, :, None, None]

        h05 = h05 + t5

        # [batch x 256 x 256 x 256]
        h06 = self.downsamp3(h05)
        # [batch x 384 x 128 x 128]
        h06 = self.norm6(h06)
        h06 = F.silu(h06)

        t6 = self.time_emb6(t)
        t6 = t6[:, :, None, None]

        h06 = h06 + t6

        # [batch x 384 x 128 x 128]
        h07 = self.downsamp4(h06)
        # [batch x 512 x 64 x 64]
        h07 = self.norm7(h07)
        h07 = F.silu(h07)

        t7 = self.time_emb7(t)
        t7 = t7[:, :, None, None]

        h07 = h07 + t7

        # batch x 512 x 64 x 64
        latent = self.bottleneck(h07)
        # batch x 512 x 64 x 64

        # upsampling

        h08 = self.norm7(latent)
        h08 = F.silu(h08)
        # batch x 512 x 64 x 64
        h08 = self.upsamp1(h08)
        # batch x 384 x 128 x 128
        h08 = h08 + t6
            
        # skip connection
        h08 = torch.concat([h08, h06], dim=1)
        # batch x 768 x 128 x 128
        h08 = self.proj1(h08)
        # batch x 384 x 128 x 128

        h09 = self.norm6(h08)
        h09 = F.silu(h09)
        # batch x 384 x 128 x 128
        h09 = self.upsamp2(h09)
        # batch x 256 x 256 x 256
        h09 = h09 + t5

        # skip connection
        h09 = torch.concat([h09, h05], dim=1)
        # batch x 512 x 256 x 256
        h09 = self.proj2(h09)
        # batch x 256 x 256 x 256

        h10 = self.norm5(h09)
        h10 = F.silu(h10)
        # batch x 256 x 256 x256
        h10 = self.upsamp3(h10)
        # batch x 192 x 512 x 512
        h10 = h10 + t4

        # skip connection
        h10 = torch.concat([h10, h04], dim=1)
        # batch x 384 x 512 x 512
        h10 = self.proj3(h10)
        # batch x 192 x 512 x 512

        h11 = self.norm4(h10)
        h11 = F.silu(h11)
        # batch x 192 x 512 x 512
        h11 = self.upsamp4(h11)
        # batch x 128 x 1024 x 1024
        h11 = h11 + t3

            # skip connection
        h11 = torch.concat([h11, h03], dim=1)
        # batch x 256 x 1024 x 1024
        h11 = self.proj4(h11)
        # batch x 128 x 1024 x 1024

        h12 = self.norm3(h11)
        h12 = F.silu(h12)
        # batch x 128 x 1024 x 1024
        h12 = self.decoder1(h12)
        # batch x 64 x 1024 x 1024
        h12 = h12 + t2

        h13 = self.norm2(h12)
        h13 = F.silu(h13)
        # batch x 64 x 1024 x 1024
        h13 = self.decoder2(h13)
        # batch x 24 x 1024 x 1024
        h13 = h13 + t1


        h14 = self.norm1(h13)
        h14 = F.silu(h14)
        h14 = self.decoder3(h14)
        # batch x 3 x 1024 x 1024
            

        return h14














        

        

Datasets Section

In [4]:

class RESCLabel(StrEnum):
    NORMAL = 'NORMAL'
    DME = 'DME'
    AMD ='AMD'
    CNV = 'CNV'
    MH = 'MK'
    DRUSEN = 'DRUSEN'
    ALL = '*'

class SplitLabel(StrEnum):
    TEST = 'test'
    TRAIN = 'train'
    VAL = 'val'

class RESCDataset(Dataset):

    def __init__(self, 
                 root_dir: str = 'datasets/retinal-oct', 
                 split:  SplitLabel= SplitLabel.TRAIN,
                 label: RESCLabel = RESCLabel.NORMAL, 
                 transform: Tensor | None = None):

        label = '*' if label is None else label
        self.paths = sorted(Path(root_dir).glob(f'{split}/{label}/*.jpg'))
        classes = sorted({p.parent for p in self.paths})
        self.class_index = {v: i for i, v in enumerate(classes)}
        self.transform = transform or transforms.ToTensor()
        self.split = split

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index: int):
        p = self.paths[index]
        image = Image.open(p).convert('RGB')
        return self.transform(image), self.class_index[p]

In [5]:
dat = RESCDataset(split='train')
len(dat)

2300

In [6]:
def train(
        loader: DataLoader,
        epochs: int,
        T: int,
        beta_min: float,
        beta_max: float,
        lr: float,
        checkpoint_dir: str,
        device: str ='cuda'
):
    assert torch.cuda.is_available(), 'GPU not vailable, use device="cpu"'
    assert 0 <= beta_min < beta_max < 1,'beta_min must be less than beta_max, and their range is [0, 1)'
    # assert 0 <= noise <= 1, 'the value of noise has a range of [0, 1]' 

    betas = torch.linspace(
        beta_min,
        beta_max,
        T,
        device=device
    )

    alpha_bar = torch.cumprod(1-betas, dim=0)

    model = UNet()
    model.to(device)

    checkpoint = Path(checkpoint_dir)

    try:
        state_dict = torch.load(
            f=checkpoint,
            map_location=device,
            weights_only=True
        )

        model.load_state_dict(state_dict=state_dict)

    except Exception as e:
        exc = type(e)
        raise exc('[Checkpoint failure] ' % e) from e

    optimizer = optim.Adam(
        params=model.parameters,
        lr=lr
    )

    model.train()

    loader_size = len(loader)

    for epoch in tqdm(range(epochs), desc=f'Training {epochs} epochs'):

        running_loss = 0.0

        for x0 in loader:

            x0.to(device)
            # batch of current dataset
            batch = x0.shape[0]
            # random timestep
            ts = torch.randint(
                low=0,
                high=T,
                size=(batch,),
                device=device
            )
            # shape -> [b]
            alpha_bar_t: Tensor = alpha_bar[ts]
            # shape -> [b, 1, 1, 1]
            alpha_bar_t = alpha_bar_t[:, None, None, None]
            # noise
            epsilon: Tensor = torch.randn_like(x0)
            # noisy data
            xt: Tensor = (alpha_bar_t.sqrt() * x0) + ((1 - alpha_bar_t).sqrt() * epsilon)
            #
            tmodel = ts.float()[:, None]/(T - 1)
            # predicted noise
            epsilon_pred = model(xt, tmodel)
            loss = F.mse_loss(epsilon_pred, epsilon)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

        average_loss = running_loss / loader_size

        print(
            f'Epoch {epoch + 1}, / {epochs}',
            f'Loss: {average_loss:.6f}'
        )

        torch.save(model.parameters(), checkpoint)

        return model





    
    



In [7]:
@torch.inference_mode()
def test(
    loader: Dataset,
    beta_min: float,
    beta_max: float,
    T: int,
    checkpoint_dir: str,
    device: str='cuda'
):
    assert torch.cuda.is_available(), 'GPU not available, use device="cpu"'
    assert 0 <= beta_min < beta_max < 1,'beta_min must be less than beta_max, and their range is [0, 1)'
    assert 0 <= noise <= 1, 'the value of noise has a range of [0, 1]'  

    betas = torch.linspace(
        start=beta_min,
        end=beta_max,
        steps=T,
        device=device
    )

    alpha_bar = torch.cumprod(1-betas, dim=0)

    




$\boxed{O=\left\lfloor\frac{I+2P-K}{S}\right\rfloor+1}$


- \(I\): input size (height or width)
- \(O\): output size
- \(P\): padding on each side
- \(K\): kernel size
- \(S\): stride

In [8]:
    # padding
    # valid = 0
    # same = 1

def conv_size(input: int, k:int, pad: int = 0, s:int=1):
    """
    input = in-channel
    k = kernel
    s = stride
    pad = padding
    """
    assert  k % 2, 'an odd number is preferred'

    return ((input + (2 * pad) - k) // s) + 1


def transpose_size(input, k, pad, s = 1, out=0, d=1):
    """
    input = in-channel
    k = kernel
    s = strise
    pad = padding
    oot = out-padding
    d =dilation
    """
    assert  k % 2, 'an odd number is preferred'

    return (input - 1) * s - (2 * pad) - (d * (k - 1)) + out + 1

In [9]:
conv_size(input=1024, k=3 ,pad=1, s=2)


512

In [10]:
384 *2

768